# Block 3 CPU smoke: classification contract

Техническая проверка без сети, GPU и учебных данных. Ноутбук проверяет фиксированный split, простой baseline, метрику и журнал эксперимента. Это не задание и не эталонное решение лабораторной.

In [ ]:
import json
import random
import tempfile
from pathlib import Path

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
assert random.Random(SEED).random() == random.Random(SEED).random()

In [ ]:
rng = np.random.default_rng(SEED)
centers = np.asarray([[-2.0, -2.0], [2.0, -2.0], [0.0, 2.0]], dtype=np.float32)
labels = np.repeat(np.arange(3), 120)
features = centers[labels] + rng.normal(0.0, 0.35, size=(len(labels), 2))

permutation = rng.permutation(len(labels))
train_idx = permutation[:240]
val_idx = permutation[240:300]
test_idx = permutation[300:]

assert not set(train_idx) & set(val_idx)
assert not set(train_idx) & set(test_idx)
assert not set(val_idx) & set(test_idx)
assert sorted(np.unique(labels[train_idx]).tolist()) == [0, 1, 2]

In [ ]:
class_centroids = np.stack([
    features[train_idx][labels[train_idx] == class_id].mean(axis=0)
    for class_id in range(3)
])

distances = ((features[test_idx, None, :] - class_centroids[None, :, :]) ** 2).sum(axis=2)
predictions = distances.argmin(axis=1)
accuracy = float(np.mean(predictions == labels[test_idx]))

assert predictions.shape == labels[test_idx].shape
assert accuracy > 0.95
print({"accuracy": accuracy, "test_size": len(test_idx)})

In [ ]:
with tempfile.TemporaryDirectory(prefix="block3-smoke-") as directory:
    journal_path = Path(directory) / "runs.jsonl"
    record = {
        "seed": SEED,
        "metrics": {"accuracy": accuracy},
        "config": {"baseline": "nearest-centroid", "split": [240, 60, 60]},
        "tags": ["block3", "cpu-smoke"],
    }
    journal_path.write_text(json.dumps(record, sort_keys=True) + "\n", encoding="utf-8")
    persisted = json.loads(journal_path.read_text(encoding="utf-8").strip())
    assert persisted["seed"] == SEED
    assert persisted["metrics"]["accuracy"] == record["metrics"]["accuracy"]

print("Block 3 CPU smoke: OK")